<!--
SPDX-FileCopyrightText: Copyright (c) 2026 NVIDIA CORPORATION & AFFILIATES. All rights reserved.
SPDX-License-Identifier: Apache-2.0

Licensed under the Apache License, Version 2.0 (the "License");
you may not use this file except in compliance with the License.
You may obtain a copy of the License at

http://www.apache.org/licenses/LICENSE-2.0

Unless required by applicable law or agreed to in writing, software
distributed under the License is distributed on an "AS IS" BASIS,
WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
See the License for the specific language governing permissions and
limitations under the License.
-->

# Notebook 02: Sequential Preprocessing and Tokenization

We build a **domain-specific tokenizer** for the 16-column transfer schema and generate
the pretraining corpora. Each customer account becomes one sequence of transactions, so
the model can learn behaviour over time (payment rhythm, recurring counterparties,
sudden shifts) instead of scoring rows in isolation.

1. **Tokenizer design** -- which column gets which tokenizer, and why.
2. **Fit once on train** -- amount bins and frequency vocabularies are data-driven.
3. **Corpus generation** -- one `<bos> ... <sep> ... <eos>` line per account chunk.

---

### What changed vs the TabFormer blueprint

The card-transaction tokenizer had a **fully static** vocabulary (known MCC list, US
states, fixed dollar thresholds), so every split could rebuild it from configuration.
This schema has no such catalogue: amount bins, BICs, countries, movement codes and
employees are learned **from the training split**. Consequences:

- The pipeline is fitted **once, on train only**, then saved
  (`tokenizer_state.json`). Val/test/inference **reload** it. Re-fitting per split
  would give the same value a different token ID in each split -- a silent bug that
  still trains and still produces embeddings, just meaningless ones.
- `vocab_size` is only known after fitting; notebook 03 reads it from
  `corpus_report.json`.
- Unseen values at inference time fall into `*_OTHER`, so the vocabulary stays closed.

In [ ]:
import warnings
warnings.filterwarnings("ignore", category=FutureWarning, module="torch.cuda")

import json
import sys
import time
from dataclasses import asdict
from pathlib import Path

import cudf

PROJECT_ROOT = Path(".").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from src.bank_corpus import fit_pipeline, generate_corpus
from src.tokenizer import (
    BankSchema,
    BankTokenizerConfig,
    BankTransactionTokenizer,
    BankTransferTokenizerPipeline,
)

from src.experiment import shared_data_dir
DATA_DIR = shared_data_dir()          # set in configs/paths.yaml (the only place)
SPLIT_DIR = DATA_DIR / "temporal_split"
CORPUS_DIR = DATA_DIR / "decoder_corpus"
TOKENIZER_DIR = DATA_DIR / "tokenizer"
# Run by scripts/run_experiment.py: every path comes from the run folder.
# Interactive use (LTM_EXPERIMENT unset): EXP is None and nothing changes.
from src.experiment import active_experiment
EXP = active_experiment()
if EXP is not None:
    globals().update(EXP.paths())
CORPUS_DIR.mkdir(parents=True, exist_ok=True)

SPLITS = {name: SPLIT_DIR / f"{name}.parquet" for name in ("train", "val", "test")}
for name, path in SPLITS.items():
    assert path.exists(), f"Missing split: {path}  (run notebook 01 first)"

SCHEMA_PATH = TOKENIZER_DIR / "schema.json"
assert SCHEMA_PATH.exists(), f"Missing {SCHEMA_PATH}  (run notebook 01 first)"
SCHEMA = BankSchema(**json.loads(SCHEMA_PATH.read_text()))

STATE_PATH = TOKENIZER_DIR / "tokenizer_state.json"
VOCAB_PATH = TOKENIZER_DIR / "vocab.json"
REPORT_PATH = CORPUS_DIR / "corpus_report.json"
CONTEXT_WINDOW = 4096   # pretraining sequence length (tokens); <= max_position_embeddings
CHUNK_COPIES = 1        # >1: overlapping train chunks, each transaction in that many (val unchanged)
PRETRAIN_ON_TEST = False  # True: the test period's sequences are added to the train corpus
if EXP is not None:
    globals().update(EXP.corpus_settings())   # CONTEXT_WINDOW, CHUNK_COPIES, PRETRAIN_ON_TEST

print(f"Splits     : {SPLIT_DIR}")
print(f"Corpora    : {CORPUS_DIR}")
print(f"Tokenizer  : {STATE_PATH}")

---

## 1. Tokenizer Design

A general-purpose BPE tokenizer fragments tabular values into meaningless subwords:
`1234.56` becomes `['123', '4', '.', '56']` (magnitude lost), an IBAN becomes a dozen
digit pieces (identity lost). We map **one field to one token** instead:

| Column | Tokenizer | Token | Rationale |
|---|---|---|---|
| `amt_euro` | `QuantileBinTokenizer` | `AMT_0..31` | continuous; quantile bins fitted on train spread a skewed amount distribution evenly, unlike fixed thresholds |
| `datetime` | `TimeDeltaTokenizer` | `TDIF_0..47` | gap since the previous transaction of the sequence, log-binned: each bin is ~1.50x the previous over a 10-year horizon, so seconds get fine bins and years coarse ones; longer gaps clip to the top bin |
| `dc_flag` | `FrequencyVocabTokenizer` | `DC_D`, `DC_C` | debit / credit |
| `movement_sign` | `FrequencyVocabTokenizer` | `SIGN_*` | a few values |
| `movement_code` | `FrequencyVocabTokenizer` | `MVC_*` | the MCC analogue; the head keeps its own token |
| `counterparty_account` | `CategoricalHashTokenizer` | `CPTY_0..4095` | very many distinct values; hashing bounds the vocabulary and pseudonymizes the account number |
| `counterparty_bic` | `FrequencyVocabTokenizer` | `BIC_*` | frequent banks keep their own token; rare ones go to the tail tokens |
| `counterparty_country` | `FrequencyVocabTokenizer` | `CTRY_*` | country code |
| `operations_status` | `FrequencyVocabTokenizer` | `OPST_*` | status flag |
| `comment_structure` | `FrequencyVocabTokenizer` | `CMT_*` | a few values |
| `employee` | `FrequencyVocabTokenizer` | `EMP_0..N` | ordinal by default so no employee identity enters the vocabulary |
| `hour`, `dow`, `month` | `FixedVocabTokenizer` | `HOUR_09`, `DOW_2`, `MONTH_04` | bounded integer ranges, no fitting needed |
| `account_number` | `CategoricalHashTokenizer` | `ACCTH_0..1023` | which account a transaction is on (customer-scoped sequences); `account_token: order` gives `ACCT_01..16` instead |

**`customer_id` emits no token.** With many customers, a per-customer token would add
as many rows to the embedding table, let the model memorize
customers instead of behaviour, and generalize to nothing for a customer unseen in
training. It is the **sequence key** instead -- identity enters the model as *which
sequence you are in*, not as a symbol.

### One sequence per customer, or per account?

A customer holding several accounts is the normal case, and
`BankTokenizerConfig.sequence_scope` decides what a sequence is:

| Scope | Sequence | Trade-off |
|---|---|---|
| `"customer"` (default) | all accounts of one customer, interleaved chronologically, each transaction carrying an `ACCTH` token | Longer sequences fill the context window better; cross-account behaviour (own-account transfers, activity spread across accounts) becomes visible; a newly opened account inherits the customer's history. Costs one token per transaction and mixes accounts that may serve different purposes. |
| `"account"` | one per `(customer_id, account_number)`, like TabFormer's `(user, card)` | Each sequence is a single behavioural unit. Shorter histories, and behaviour that only shows up across accounts is invisible. |

Transactions are never mixed across customers under either setting. Everything
downstream -- sorting, the inter-transaction gap, corpus chunking, and the history
windows in notebook 04 -- reads `pipeline.group_cols`, so this is the only switch.

With customer-scoped sequences the `TDIF` gap is measured along the customer's whole
timeline. Set `include_account_time_delta=True` to also emit `TDIFA`, the gap within the
account (one extra token per transaction).

**Why hash the counterparty and the account, but rank BICs by frequency?** Hashing mixes
unrelated values at every frequency level, which is acceptable for a very high-cardinality
field where most values appear once or twice, and it is *split-stable* a rank of the
account within the customer would shift between splits whenever an account has no
transactions in one of them, silently changing the token for the same account. For BICs
the head of the distribution is small and highly informative, so it keeps
collision-free tokens and only the rare tail is pooled.

In [ ]:
# ======================================================================
# EDIT HERE: vocabulary caps and derived-feature switches.
# Caps are upper bounds -- a field with fewer distinct values stays smaller.
# ======================================================================
CONFIG = BankTokenizerConfig(
    sequence_scope="customer",      # "customer" (accounts interleaved) | "account"
    include_account_time_delta=True,  # True -> extra TDIFA gap within the account
    drop_fields=("EMP",),             # token prefixes left out of the sequence (column still read)
    amount_bins=32,
    amount_clip_quantiles=(0.0005, 0.9995),
    include_time_delta=True,
    time_delta_bins=48,             # ~1.50x per bin over a 10-year horizon
    time_delta_max_years=2.5,      # set to your extract's span; longer gaps clip
    counterparty_hash_size=4096,
    bic_max_vocab=2048,
    country_max_vocab=256,
    employee_max_vocab=128,
    employee_readable=False,        # True -> EMP_<name>; keep False to stay pseudonymous
    comment_structure_max_vocab=16,
    operations_status_max_vocab=4,
    movement_code_max_vocab=512,
    movement_sign_max_vocab=8,
    dc_flag_max_vocab=4,
    movement_group_divisor=None,    # e.g. 100 if movement codes are hierarchical
    derive_time_parts=True,         # recompute hour/dow/month from datetime
    datetime_format=None,           # e.g. "%Y-%m-%d %H:%M:%S" if inference is slow/wrong
    # Free text (SCHEMA.description). None -> no text field. "words" | "bag" | "category" |
    # "hashed" -- see src/tokenizer/text.py. Adds text_slots tokens per transaction.
    text_strategy=None,
    text_slots=4,
    text_min_customers=5,           # a word needs this many distinct customers (keeps names out)
    # Values without a token of their own (src/tokenizer/tail.py): "rare_bins" (rarity
    # bands), "hash" (buckets) or "other" (the original single OTHER token, whose
    # embedding never trains when the cap exceeds the field's cardinality).
    tail_strategy="rare_bins",
    tail_min_count=10,              # values seen fewer times in train use the tail tokens
    # New fields, by token prefix (src/tokenizer/field_kinds.py). Kinds: "categorical"
    # (one string/code per row), "identifier" (10^4+ IDs), "numeric", "text". E.g.
    #   {"CHAN": {"column": "channel", "kind": "categorical"}}
    # Unsure which kind? suggest_kind(df["column"]) from src.tokenizer says why.
    extra_fields={},
)
# ======================================================================

if EXP is not None:
    CONFIG = EXP.tokenizer_config(CONFIG)   # the experiment's tokenizer settings win

preview = BankTransferTokenizerPipeline(schema=SCHEMA, config=CONFIG)
if EXP is not None:
    EXP.check_loss_weights(preview)   # fail now, not after hours of tokenizing
print(preview.describe())
print(
    f"\nTokens per transaction : {preview.tokens_per_transaction}\n"
    f"Transactions per {CONTEXT_WINDOW}-token sequence: "
    f"{preview.recommended_chunk_size(CONTEXT_WINDOW)}"
)

### 1.1 Fit on the training split

One pass over `train.parquet`: `preprocess()` derives the columns the steps consume
(amount as float, inter-transaction gap in seconds, hashed counterparty and account,
cleaned strings, calendar parts) and sorts by `(*group_cols, datetime)`. Then `fit()`
learns the amount bin edges and the frequency vocabularies.

In [ ]:
t0 = time.time()
pipeline, train_processed = fit_pipeline(
    SPLITS["train"],
    schema=SCHEMA,
    config=CONFIG,
    state_path=STATE_PATH,
    vocab_path=VOCAB_PATH,
)
print(f"\nTotal fit time: {time.time() - t0:.1f}s")
print(f"vocab_size: {pipeline.global_vocab_size}")

In [ ]:
print("Fitted vocabulary per field:")
print(f"{'step':<16} {'tokenizer':<26} {'size':>6} {'offset':>7}  examples")
print("-" * 110)
for tok_id, info in pipeline.token_legend(6).items():
    print(
        f"{tok_id:<16} {info['step']:<26} {info['vocab_size']:>6} "
        f"{info['offset']:>7}  {', '.join(map(str, info['examples']))}"
    )
print("-" * 110)
print(f"{'special':<16} {'<pad> <bos> <eos> <sep> <unk>':<26} {pipeline.num_special_tokens:>6} {0:>7}")
print(f"\nTotal vocab_size = {pipeline.global_vocab_size}")

amt_step = pipeline.steps["amt_val"]
print(f"\nAmount bin edges ({amt_step.used_bins} bins used of {amt_step.num_bins}):")
print("  " + ", ".join(f"{e:,.2f}" for e in amt_step.edges[:12]) + (" ..." if len(amt_step.edges) > 12 else ""))

### 1.2 Is the time-gap horizon right for your data?

`TDIF` spreads its bins over the **log** of `time_delta_max_years`, so resolution is
multiplicative: `step = horizon ** (1 / bins)`. That has two consequences worth
internalizing before tuning it:

- **A longer horizon is nearly free.** At a fixed 1.5x resolution, going from a 1-year
  to a 10-year horizon costs 6 bins. Shrinking the horizon to "save" bins barely helps:
  10 years over 32 bins is 1.84x per bin, 2 years over 32 bins is 1.75x.
- **Clipping is what a short horizon really buys you.** Every gap above the horizon
  collapses into the top bin. With a 2-year horizon and 32 bins, everything from ~8
  months upward is one token.

So: set the horizon to the span of your extract (notebook 01 printed it) and buy
resolution with `time_delta_bins`. The default -- 10 years, 48 bins, ~1.50x per bin --
costs 16 more vocabulary rows than 32 bins, out of ~8,000.

The cell below checks that against your actual gaps: if the top bins are empty, the
horizon is longer than your data needs; if a noticeable share clips, it is too short.

In [ ]:
import math

if "time_delta_s" in train_processed.columns:
    tdif = pipeline.steps["time_delta_s"]
    gaps = train_processed["time_delta_s"]

    def human(seconds):
        for div, unit in ((1, "s"), (60, "min"), (3600, "h"), (86400, "d")):
            if seconds < div * 90 or unit == "d":
                if unit == "d" and seconds >= 86400 * 400:
                    break
                return f"{seconds / div:,.1f}{unit}"
        return f"{seconds / 31556952:,.2f}y"

    step_ratio = math.exp(math.log(tdif.max_horizon + 1.0) / tdif.num_bins)
    print(f"Horizon    : {tdif.max_years} years ({tdif.max_horizon:,} s)")
    print(f"Bins       : {tdif.num_bins}  ->  {step_ratio:.3f}x per bin")

    probs = [0.5, 0.9, 0.99, 0.999, 1.0]
    qs = gaps.quantile(probs).to_pandas().tolist()
    print("\nObserved gaps between consecutive transactions of a sequence:")
    for p, v in zip(probs, qs):
        print(f"  p{p * 100:<6g}: {human(float(v)):>12}")

    clipped = float((gaps > tdif.max_horizon).mean())
    print(f"\nGaps beyond the horizon (folded into the top bin): {clipped:.4%}")

    counts = tdif.tokenize(gaps).value_counts().to_pandas().to_dict()
    log_max = math.log(tdif.max_horizon + 1.0)
    total = len(gaps)
    used = 0
    print(f"\n{'bin':>5} {'from':>12} {'share':>8}")
    for i in range(tdif.num_bins):
        n = int(counts.get(f"TDIF_{i}", 0))
        if not n:
            continue
        used += 1
        lower = math.exp(log_max * i / tdif.num_bins) - 1.0
        bar = "#" * max(1, round(40 * n / total))
        print(f"{i:>5} {human(lower):>12} {n / total:>7.2%}  {bar}")
    print(f"\nBins carrying data: {used}/{tdif.num_bins}")

    if clipped > 0.01:
        print("  -> more than 1% of gaps clip; raise time_delta_max_years.")
    elif tdif.num_bins - used > 6:
        print(f"  -> {tdif.num_bins - used} bins are unused. Lower time_delta_max_years "
              "to your data's span (or spend the bins elsewhere); resolution improves "
              "only slightly, so this is cosmetic unless the tail is large.")
    else:
        print("  -> horizon and bin count fit the data.")
else:
    print("include_time_delta=False: no TDIF token in this configuration.")

### 1.2 A single transaction, tokenized

Real rows from the training split -- no synthetic example, so what you see is what the
model will train on.

In [ ]:
sample = train_processed.head(3)
token_sample = pipeline.transform(sample).to_pandas()

for row in range(len(token_sample)):
    fields = token_sample.iloc[row].to_dict()
    print(f"Transaction {row}:")
    for k, v in fields.items():
        print(f"  {k:<16} -> {v}")
    print(f"  sequence: <bos> {' '.join(str(v) for v in fields.values())} <eos>\n")

### 1.3 Comparison with a general-purpose BPE tokenizer

The token count per transaction decides how much history fits in the context window.

In [ ]:
from transformers import AutoTokenizer

# Illustration only: needs the GPT-2 tokenizer from huggingface.co (or the local cache).
# On a machine without internet the comparison is skipped; nothing downstream uses it.
try:
    gpt2 = AutoTokenizer.from_pretrained("gpt2")
except OSError as exc:
    gpt2 = None
    print(f"GPT-2 tokenizer not available ({type(exc).__name__}: no internet / not cached); "
          "comparison skipped.")

if gpt2 is not None:
    raw_row = train_processed.head(1)[SCHEMA.feature_columns].to_pandas().iloc[0]
    raw_text = ", ".join(str(v) for v in raw_row.values)
    gpt2_tokens = gpt2.tokenize(raw_text)

    fin_tokens = list(token_sample.iloc[0].values)
    fin_count = len(fin_tokens)
    gpt2_count = len(gpt2_tokens)

    fin_per_seq = (CONTEXT_WINDOW - 2) // (fin_count + 1)
    gpt2_per_seq = (CONTEXT_WINDOW - 2) // (gpt2_count + 1)

    print("=" * 78)
    print("TOKENIZER COMPARISON: one transaction, two approaches")
    print("=" * 78)
    print(f"\n--- Bank tokenizer (domain-specific pipeline) ---")
    print(f"Count : {fin_count} tokens")
    print(f"Tokens: {fin_tokens}")
    print(f"\n--- GPT-2 BPE on the raw serialized row ---")
    print(f"Input : {raw_text[:160]}...")
    print(f"Count : {gpt2_count} tokens")
    print(f"Tokens: {gpt2_tokens[:24]} ...")
    print("\n" + "-" * 78)
    print(f"{'Tokenizer':<20} {'Tokens/txn':>12} {'Compression':>16} {f'Txns in {CONTEXT_WINDOW}':>16}")
    print("-" * 78)
    print(f"{'Bank (domain)':<20} {fin_count:>12} {'1x (baseline)':>16} {f'~{fin_per_seq}':>16}")
    print(f"{'GPT-2 (BPE)':<20} {gpt2_count:>12} {f'{gpt2_count / fin_count:.1f}x more':>16} {f'~{gpt2_per_seq}':>16}")
    print("-" * 78)
    print(
        f"\nSame context window, {fin_per_seq / max(gpt2_per_seq, 1):.1f}x more history per "
        "sequence -- and every token is a whole financial concept rather than a digit fragment."
    )

### 1.4 Edge cases

Three behaviours worth confirming on your data: extreme amounts land in the outer
quantile bins, the first transaction of a sequence lands in the lowest gap bin
(`TDIF_1` -- a zero gap sits above the first boundary, so `TDIF_0` is never emitted),
and values unseen during fitting fall into `*_OTHER` instead of breaking the vocabulary.

In [ ]:
amt_col = "amt_val"
extremes = cudf.concat(
    [
        train_processed.nsmallest(1, amt_col),
        train_processed.nlargest(1, amt_col),
    ]
)
extreme_tokens = pipeline.transform(extremes).to_pandas()
print("Amount extremes:")
for amount, token in zip(
    extremes[amt_col].to_pandas().values, extreme_tokens[amt_col].values
):
    print(f"  {amount:>18,.2f} EUR -> {token}")

if "time_delta_s" in train_processed.columns:
    first_of_account = train_processed[train_processed["time_delta_s"] == 0].head(1)
    if len(first_of_account):
        tokens = pipeline.transform(first_of_account).to_pandas().iloc[0]
        if "time_delta_s" in tokens:   # missing when the field is dropped (drop_fields)
            print(f"\nFirst transaction of an account: {tokens['time_delta_s']} (zero gap)")

unseen = train_processed.head(1).copy()
unseen["bic_clean"] = "ZZZZUNSEENBIC"
unseen["ctry_clean"] = "ZZ"
unseen_tokens = pipeline.transform(unseen).to_pandas().iloc[0]
print()
for col, value in (("bic_clean", "ZZZZUNSEENBIC"), ("ctry_clean", "ZZ")):
    if col in unseen_tokens:   # missing when the field is dropped (drop_fields)
        print(f"Unseen {value!r:16s} -> {unseen_tokens[col]}")

---

## 2. Generate the Pretraining Corpora

Every split is transformed with the **fitted** pipeline (val/test are never re-fitted)
and written as text: one line per (sequence, chunk) with `<sep>` between transactions.

```
<bos> AMT_12 TDIF_7 DC_D SIGN_NEG MVC_17 CPTY_1183 BIC_42 CTRY_3 OPST_OK CMT_3 EMP_7 HOUR_09 DOW_2 MONTH_04 ACCTH_318 <sep> AMT_5 ... <eos>
```

`chunk_size` fills the context window: with 15 tokens per transaction plus one
separator, ~255 transactions fit in 4096 tokens. A customer (or account, under
`sequence_scope="account"`) with more history produces several lines; one with less
produces a single shorter line.

In [ ]:
CHUNK_SIZE = pipeline.recommended_chunk_size(CONTEXT_WINDOW)
print(f"Sequence scope: {pipeline.config.sequence_scope} -> key {pipeline.group_cols}")
print(f"Chunk size    : {CHUNK_SIZE} transactions per sequence "
      f"({pipeline.tokens_per_transaction} tokens/txn + 1 separator)\n")

from src.fingerprint import file_fingerprint

report = {
    "vocab_size": int(pipeline.global_vocab_size),
    "tokens_per_transaction": pipeline.tokens_per_transaction,
    "sequence_scope": pipeline.config.sequence_scope,
    "sequence_key": list(pipeline.group_cols),
    "chunk_size": CHUNK_SIZE,
    "chunk_copies": CHUNK_COPIES,
    "context_window": CONTEXT_WINDOW,
    "tokenizer_state": str(STATE_PATH),
    # Content, not path: NB04 refuses a checkpoint trained under another tokenizer.
    "tokenizer_fingerprint": file_fingerprint(STATE_PATH),
    "vocab_path": str(VOCAB_PATH),
    "schema": asdict(SCHEMA),
    "config": asdict(pipeline.config),
    "splits": {},
}

for name, path in SPLITS.items():
    corpus_path = CORPUS_DIR / f"{name}_corpus.txt"
    # train_processed is already preprocessed; val/test are read and preprocessed here.
    source = train_processed if name == "train" else path
    report["splits"][name] = generate_corpus(
        pipeline, corpus_path, source,
        chunk_size=CHUNK_SIZE, context_window=CONTEXT_WINDOW,
        # Overlap only where it adds training signal; val/test stay one chunk per
        # stretch of history, so validation loss remains comparable across runs.
        chunk_copies=CHUNK_COPIES if name == "train" else 1,
    )

# Pretraining uses no labels, so the test period can train it too (val stays held out:
# it decides the early stop). The model has then seen the test period's transfers:
# fine for the customer task (NB07); NB05's transfer-level test scores are no longer
# on unseen data -- compare them only with runs that do the same.
report["pretrain_on_test"] = bool(PRETRAIN_ON_TEST)
if PRETRAIN_ON_TEST:
    import shutil
    test_stats = report["splits"]["test"]
    with open(CORPUS_DIR / "train_corpus.txt", "a") as out, open(test_stats["path"]) as src:
        shutil.copyfileobj(src, out)
    report["splits"]["train"]["sequences"] += test_stats["sequences"]
    report["splits"]["train"]["rows"] += test_stats["rows"]
    print(f"\nPRETRAIN_ON_TEST: {test_stats['sequences']:,} test sequences added to "
          f"train_corpus.txt (now {report['splits']['train']['sequences']:,})")

REPORT_PATH.write_text(json.dumps(report, indent=2))
print(f"\nReport -> {REPORT_PATH}")

## 3. Verify the Output

Two checks that catch the mistakes that matter: (a) the corpus decodes back through the
**reloaded** tokenizer state, proving train and inference agree on token IDs, and
(b) no sequence exceeds the context window.

In [ ]:
import os

print("Corpus summary:")
print("=" * 74)
for name in SPLITS:
    path = CORPUS_DIR / f"{name}_corpus.txt"
    n_lines = sum(1 for _ in open(path))
    size_mb = os.path.getsize(path) / (1024 * 1024)
    stats = report["splits"][name]
    print(
        f"  {name:<6}: {n_lines:>9,} sequences  ({size_mb:>8.1f} MB)  "
        f"max {stats['longest_sequence_tokens']:>5} tokens  {path.name}"
    )

with open(CORPUS_DIR / "train_corpus.txt") as f:
    first_line = f.readline().strip()
print(f"\nFirst training sequence ({len(first_line.split())} tokens, "
      f"{first_line.count('<sep>') + 1} transactions):")
print(f"  {first_line[:300]}...")

In [ ]:
tokenizer = BankTransactionTokenizer.from_state_file(STATE_PATH)
print(f"Reloaded tokenizer: vocab_size={tokenizer.vocab_size}, "
      f"{tokenizer.tokens_per_transaction} tokens/transaction")

ids = tokenizer.encode(first_line)
unknown = [
    tok for tok in first_line.split() if tok not in tokenizer.vocab
]
print(f"Encoded {len(ids)} ids; max id = {max(ids)} (must be < vocab_size)")
print(f"Unknown tokens in corpus: {len(unknown)}")
assert not unknown, f"Corpus contains tokens outside the vocabulary: {unknown[:10]}"
assert max(ids) < tokenizer.vocab_size

roundtrip = tokenizer.decode(ids)
assert roundtrip.split() == first_line.split(), "decode(encode(x)) != x"
print("Round-trip encode/decode: OK")
print(f"\nvocab_size to pass to notebook 03: {report['vocab_size']}")

---

## Summary

The tokenizer emits **15 tokens per transaction** (14 with
`sequence_scope="account"`) and was fitted once on the training split;
`tokenizer_state.json` is now the single source of truth for token IDs, the schema and
the sequence scope, and notebooks 03-05 read all of it from there.

**Outputs**

| File | Used by |
|---|---|
| `decoder_corpus/{train,val,test}_corpus.txt` | NB03 (pretraining) |
| `decoder_corpus/corpus_report.json` | NB03 (`vocab_size`), NB04 |
| `tokenizer/tokenizer_state.json` | NB04 (transform + history windows) |
| `tokenizer/vocab.json` | NB03 (training data module) |

Continue to [03_foundation_model_training.ipynb](./03_foundation_model_training.ipynb).